# Experiments

# Load Processed Training Data

Load the preprocessed training split from the saved pipeline outputs. This ensures the hyperparameter tuning stage has the feature matrix and labels it needs.

In [8]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("..") / "data" / "processed"
PIPELINE_OUTPUT_DIR = DATA_DIR / "pipeline_outputs"

X_TRAIN_FILE = PIPELINE_OUTPUT_DIR / "X_train.csv"
X_TEST_FILE = PIPELINE_OUTPUT_DIR / "X_test.csv"
Y_TRAIN_FILE = PIPELINE_OUTPUT_DIR / "y_train.csv"
Y_TEST_FILE = PIPELINE_OUTPUT_DIR / "y_test.csv"

for input_file in [X_TRAIN_FILE, X_TEST_FILE, Y_TRAIN_FILE, Y_TEST_FILE]:
    if not input_file.exists():
        raise FileNotFoundError(
            f"Expected {input_file} to exist."
        )

X_train = pd.read_csv(X_TRAIN_FILE)
X_test = pd.read_csv(X_TEST_FILE)
y_train = pd.read_csv(Y_TRAIN_FILE).squeeze()
y_test = pd.read_csv(Y_TEST_FILE).squeeze()

# Drop identifier columns if they remain in the saved feature sets
for id_col in ["customerID", "customer_id", "id"]:
    for dataset in [X_train, X_test]:
        if id_col in dataset.columns:
            dataset.drop(columns=[id_col], inplace=True)

print("Loaded training data:", X_train.shape, y_train.shape)
print("Loaded test data:", X_test.shape, y_test.shape)

Loaded training data: (5625, 30) (5625,)
Loaded test data: (1407, 30) (1407,)


# Random Forest Hyperparameter Tuning

Use `RandomizedSearchCV` to search the Random Forest hyperparameter space using cross-validation and `f1` scoring. This section returns the best parameters and cross-validation score without evaluating on the test set.

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
}

rf_model = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print(f"Best cross-validation F1 score: {random_search.best_score_:.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 10}
Best cross-validation F1 score: 0.5786


`RandomizedSearchCV` is preferred over manual tuning because it efficiently explores many hyperparameter combinations with randomized sampling, which is faster than manually trying each setting. This approach often finds strong configurations more quickly and systematically when the search space is large.

# Evaluate Tuned Random Forest

Use the best estimator from `RandomizedSearchCV` to make test-set predictions, and compare tuned performance versus a default Random Forest and a Logistic Regression baseline.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Baseline models for comparison
baseline_lr = LogisticRegression(max_iter=1000, random_state=42)
baseline_rf = RandomForestClassifier(random_state=42)

baseline_lr.fit(X_train, y_train)
baseline_rf.fit(X_train, y_train)

# Predictions
y_pred_tuned_rf = random_search.best_estimator_.predict(X_test)
y_pred_default_rf = baseline_rf.predict(X_test)
y_pred_lr = baseline_lr.predict(X_test)

def compute_metrics(y_true, y_pred, y_scores=None):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_scores if y_scores is not None else y_pred),
    }

metrics_tuned_rf = compute_metrics(
    y_test,
    y_pred_tuned_rf,
    random_search.best_estimator_.predict_proba(X_test)[:, 1],
)
metrics_default_rf = compute_metrics(
    y_test,
    y_pred_default_rf,
    baseline_rf.predict_proba(X_test)[:, 1],
)
metrics_lr = compute_metrics(
    y_test,
    y_pred_lr,
    baseline_lr.predict_proba(X_test)[:, 1],
)

print("Tuned Random Forest Classification Report:\n")
print(classification_report(y_test, y_pred_tuned_rf))

cm_tuned_rf = confusion_matrix(y_test, y_pred_tuned_rf)
fig, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_tuned_rf)
disp.plot(ax=ax)
plt.title("Tuned Random Forest Confusion Matrix")
plt.show()

comparison_df = pd.DataFrame([
    {
        "model": "Tuned Random Forest",
        **metrics_tuned_rf,
    },
    {
        "model": "Default Random Forest",
        **metrics_default_rf,
    },
    {
        "model": "Logistic Regression",
        **metrics_lr,
    },
])

comparison_df = comparison_df.sort_values(by="f1", ascending=False).reset_index(drop=True)
print("Model comparison sorted by F1 score:")
print(comparison_df)

d:\cv projects\RetentionAI\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Tuned Random Forest Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.90      0.86      1033
           1       0.62      0.47      0.54       374

    accuracy                           0.78      1407
   macro avg       0.72      0.68      0.70      1407
weighted avg       0.77      0.78      0.77      1407



NameError: name 'plt' is not defined

The tuned Random Forest is evaluated on the test set and compared against the logistic regression baseline and the default Random Forest. The comparison table is sorted by F1 score so the best model is shown first.

If the tuned Random Forest has a higher F1 score than the default Random Forest, hyperparameter tuning improved the model. If it also exceeds the logistic regression baseline, the tuning step provided additional value over a standard linear model.